# 2025-11-06 GTX-A 광역버스 교통카드 합성데이터 수집`n
`n
분석대상 전체 노선을 조회하고 경기도 승차·서울 하차 자료를 저장한다. 기존 2024·2025 파일은 덮어쓰지 않는다.`n

In [1]:
# 프로젝트 루트(UROP) 자동 탐색 — 절대경로 없이 어느 컴퓨터에서도 동작
import sys
from pathlib import Path
UROP_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.py').is_file() and (p / 'code').is_dir())
for _d in [UROP_ROOT, *sorted(d for d in (UROP_ROOT / 'code').rglob('*') if d.is_dir() and d.name != '__pycache__')]:
    if str(_d) not in sys.path:
        sys.path.append(str(_d))

from pathlib import Path
from getpass import getpass
import time
import requests
import pandas as pd

BUS_ID_FILE = UROP_ROOT / "data" / "raw" / "gtx_a" / "route_selection" / "P_G_BUSID_24routes.csv"
OUT = UROP_ROOT / "data" / "raw" / "gtx_a" / "transport_card"
OUT.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://apis.data.go.kr/1613000/RegionalTransportationCardUsageSyntheticData/getGyeonggiTransportationCardUsageSyntheticData"
KEY = getpass("공공데이터포털 인증키를 입력하세요: ").strip()
DATES = ["20251106"]
EXCLUDE_ROUTE_NOS = {"7602", "2200-1", "1500(예약)", "M7111(예약)", "M7412(예약)", "3100N", "9709N"}
DATE_EXCLUDE_ROUTE_NOS = {"20241017": {"7111", "9030", "9030-1", "M7111"}}
DATE_ROUTE_ID_OVERRIDES = {
    "20241017": {
        "3100": "41091022",
        "2200": "41091008",
        "9030": "41091013",
        "9030-1": "41091017",
        "M7111": "41091900",
    }
}
PAGE_SIZE = 1000
RIDE_CTPV_CD = "41"  # 경기도 승차
GOFF_CTPV_COLUMNS = ["goff_ctpv_cd", "alight_ctpv_cd", "arr_ctpv_cd", "arrive_ctpv_cd"]
if not KEY:
    raise ValueError("인증키가 없습니다.")

bus_ids = pd.read_csv(BUS_ID_FILE, dtype=str, encoding="utf-8-sig").fillna("")
bus_ids = bus_ids[bus_ids["STCIS_노선ID"].str.strip().ne("")].copy()
bus_ids["STCIS_노선번호"] = bus_ids["STCIS_노선번호"].str.strip().str.lstrip("'")
bus_ids = bus_ids[~bus_ids["STCIS_노선번호"].isin(EXCLUDE_ROUTE_NOS)]
bus_ids = bus_ids.drop_duplicates("STCIS_노선ID").reset_index(drop=True)
print(f"조회 대상: {len(bus_ids)}개 분석 노선")

def parse(payload):
    r = payload.get("response", payload.get("Response", payload))
    h, b = r.get("header", {}) or {}, r.get("body", {}) or {}
    items = (b.get("items", {}) or {})
    items = items.get("item", []) if isinstance(items, dict) else items
    return ([items] if isinstance(items, dict) else items or []), h, b

def get_page(route_id, date, page):
    params = {"serviceKey": KEY, "pageNo": page, "numOfRows": PAGE_SIZE,
              "dataType": "JSON", "opr_ymd": date, "rte_id": route_id,
              "users_type_cd": "01", "ride_ctpv_cd": RIDE_CTPV_CD}
    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    items, header, body = parse(r.json())
    code = str(header.get("resultCode", header.get("resultcode", "")))
    if code not in ("", "00", "0", "200"):
        raise RuntimeError(f"resultCode={code}: {header}")
    return items, body

def collect(date):
    rows, logs = [], []
    date_bus_ids = bus_ids[~bus_ids["STCIS_노선번호"].isin(DATE_EXCLUDE_ROUTE_NOS.get(date, set()))].reset_index(drop=True)
    for i, meta in date_bus_ids.iterrows():
        rid = DATE_ROUTE_ID_OVERRIDES.get(date, {}).get(meta["STCIS_노선번호"].strip(), meta["STCIS_노선ID"].strip())
        rno = meta["STCIS_노선번호"].strip().lstrip("'")
        got, page, error = [], 1, ""
        try:
            while True:
                items, body = get_page(rid, date, page)
                got.extend(items)
                total = int(body.get("totalCount", body.get("totalcount", 0)) or 0)
                if not items or (total and len(got) >= total) or len(items) < PAGE_SIZE:
                    break
                page += 1
            for x in got:
                x.update({"query_route_id": rid, "query_route_no": rno, "query_date": date,
                          "GTX_A_역": meta.get("GTX_A_역", ""), "대상지역": meta.get("대상지역", "")})
            rows.extend(got)
            logs.append({"query_date": date, "route_id": rid, "route_no": rno, "status": "success", "count": len(got), "error": ""})
            print(f"[{i+1}/{len(date_bus_ids)}] {rno}: {len(got)}건")
        except Exception as e:
            error = repr(e)
            logs.append({"query_date": date, "route_id": rid, "route_no": rno, "status": "failed", "count": 0, "error": error})
            print(f"[{i+1}/{len(date_bus_ids)}] {rno}: 실패 - {e}")
        time.sleep(0.2)
    raw, log = pd.DataFrame(rows), pd.DataFrame(logs)
    if not raw.empty:
        goff_col = next((c for c in GOFF_CTPV_COLUMNS if c in raw.columns), None)
        if goff_col is None:
            raise KeyError(f"서울 하차 필드를 찾지 못했습니다. 응답 컬럼: {raw.columns.tolist()}")
        raw[goff_col] = raw[goff_col].astype(str).str.strip().str.zfill(2)
        raw = raw[raw[goff_col].eq("11")].copy()
        counts = raw.groupby("query_route_id").size()
        log["count"] = log["route_id"].map(counts).fillna(0).astype(int)
    filtered_counts = raw.groupby("query_route_no").size().to_dict() if not raw.empty else {}
    for meta in date_bus_ids.itertuples(index=False):
        route_no = str(getattr(meta, "STCIS_노선번호")).strip().lstrip("'")
        print(f"[서울 하차 필터 후] {route_no}: {filtered_counts.get(route_no, 0)}건")
    raw.to_csv(OUT / f"gtx_a_transport_card_{date}_raw.csv", index=False, encoding="utf-8-sig")
    log.to_csv(OUT / f"gtx_a_transport_card_{date}_query_log.csv", index=False, encoding="utf-8-sig")
    return raw, log

results = {}
for date in DATES:
    print(f"\n===== {date} =====")
    results[date] = collect(date)

# 노선별 총건수가 아니라 승차·하차 정류장 단위 OD로 저장
od_frames = []
for date, (raw_df, _) in results.items():
    if raw_df.empty:
        continue
    od = raw_df.copy()
    od["utztn_nope"] = pd.to_numeric(od["utztn_nope"], errors="coerce").fillna(0)
    od = (od.groupby(["query_date", "query_route_id", "query_route_no", "GTX_A_역", "대상지역",
                      "ride_ctpv_cd", "goff_ctpv_cd", "ride_sttn_id", "goff_sttn_id"], as_index=False)
             .agg(통행건수=("utztn_nope", "sum"), 원본행수=("utztn_nope", "size")))
    od = od.rename(columns={"ride_sttn_id": "승차정류장ID", "goff_sttn_id": "하차정류장ID"})
    od_path = OUT / f"gtx_a_transport_card_{date}_od.csv"
    od.to_csv(od_path, index=False, encoding="utf-8-sig")
    od_frames.append(od)
    print(f"OD 저장: {od_path} ({len(od)}개 승차-하차 조합)")

summary = pd.DataFrame([{"query_date": d, "routes": len(log), "success_routes": (log.status == "success").sum(),
                         "failed_routes": (log.status == "failed").sum(), "raw_rows": log["count"].sum()}
                        for d, (_, log) in results.items()])
summary.to_csv(OUT / "gtx_a_transport_card_collection_summary_20251106.csv", index=False, encoding="utf-8-sig")
display(summary)

조회 대상: 24개 분석 노선

===== 20251106 =====
[1/24] 1000: 3247건
[2/24] 1082: 877건
[3/24] 1100: 31건
[4/24] 1200: 1162건
[5/24] 1500: 1422건
[6/24] 1900: 41건
[7/24] 2200: 689건
[8/24] 3100: 349건
[9/24] 3400: 218건
[10/24] 6701: 66건
[11/24] 7101: 165건
[12/24] 9030: 310건
[13/24] 9030-1: 20건
[14/24] 9600: 41건
[15/24] 9700: 61건
[16/24] 9709: 229건
[17/24] 9710: 245건
[18/24] 9710-1: 33건
[19/24] G7426: 293건
[20/24] G7625: 193건
[21/24] M7111: 206건
[22/24] M7154: 86건
[23/24] M7412: 497건
[24/24] M7731: 1478건
[서울 하차 필터 후] 1000: 2430건
[서울 하차 필터 후] 1082: 442건
[서울 하차 필터 후] 1100: 29건
[서울 하차 필터 후] 1200: 768건
[서울 하차 필터 후] 1500: 897건
[서울 하차 필터 후] 1900: 35건
[서울 하차 필터 후] 2200: 630건
[서울 하차 필터 후] 3100: 319건
[서울 하차 필터 후] 3400: 213건
[서울 하차 필터 후] 6701: 44건
[서울 하차 필터 후] 7101: 155건
[서울 하차 필터 후] 9030: 295건
[서울 하차 필터 후] 9030-1: 19건
[서울 하차 필터 후] 9600: 41건
[서울 하차 필터 후] 9700: 58건
[서울 하차 필터 후] 9709: 170건
[서울 하차 필터 후] 9710: 148건
[서울 하차 필터 후] 9710-1: 28건
[서울 하차 필터 후] G7426: 269건
[서울 하차 필터 후] G7625: 184건
[서울 하차 필터 후] M7111: 204건
[서울

,query_date,routes,success_routes,failed_routes,raw_rows
0,20251106,24,24,0,9241
